# 02 — Data Quality & Validation

Phase 3. This is the phase that matters most for an Operations story: before any modelling, we quantify exactly what's wrong with the data, and — critically — *investigate* anomalies rather than reflexively deleting them. A large transaction amount might be the single most important row in the dataset, not noise.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('..').resolve() / 'src'))
import preprocessing

df = preprocessing.load_raw()
findings = preprocessing.validate(df)
df.shape

(284807, 31)

## Missing values — which columns, how many, what %

In [2]:
findings['missing_values']

{'total': 0, 'by_column': {}, 'pct_by_column': {}}

**Zero missing values across all 31 columns.** This is unusually clean for a real-world feed — a direct consequence of this dataset already having been through one round of institutional cleaning before public release. Nothing to impute or drop here.

## Duplicate transaction records

In [3]:
findings['duplicate_rows']

{'count': 1081, 'fraud_among_duplicates': 19}

In [4]:
dup_mask = df.duplicated(keep=False)
print(f"{dup_mask.sum()} rows involved in exact duplicates "
      f"({100*dup_mask.sum()/len(df):.3f}% of the dataset)")
df[dup_mask].sort_values(list(df.columns)).head(6)

1854 rows involved in exact duplicates (0.651% of the dataset)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
34,26.0,-0.535388,0.865268,1.351076,0.147575,0.433680,0.086983,0.693039,0.179742,-0.285642,...,0.049526,0.206537,-0.187108,0.000753,0.098117,-0.553471,-0.078306,0.025427,1.77,0
35,26.0,-0.535388,0.865268,1.351076,0.147575,0.433680,0.086983,0.693039,0.179742,-0.285642,...,0.049526,0.206537,-0.187108,0.000753,0.098117,-0.553471,-0.078306,0.025427,1.77,0
32,26.0,-0.529912,0.873892,1.347247,0.145457,0.414209,0.100223,0.711206,0.176066,-0.286717,...,0.046949,0.208105,-0.185548,0.001031,0.098816,-0.552904,-0.073288,0.023307,6.14,0
33,26.0,-0.529912,0.873892,1.347247,0.145457,0.414209,0.100223,0.711206,0.176066,-0.286717,...,0.046949,0.208105,-0.185548,0.001031,0.098816,-0.552904,-0.073288,0.023307,6.14,0
112,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0
113,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0


**Why these are treated as duplicate records, not genuine repeat transactions:** each row has 28 continuous PCA components plus `Time` and `Amount` — 30 continuous values in total. Two independent, genuinely different transactions matching on *all 30* by chance is astronomically unlikely. The far more plausible explanation is a duplicate submission or duplicate logging event upstream. 19 of the 1,081 duplicate rows are labelled fraud — keeping them would let the same fraud case appear in both the training and test split, which is a leakage risk in its own right (Phase 7 territory, encountered a phase early). They are dropped in the cleaning step below.

## Data types

In [5]:
df.dtypes

Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Every column is already numeric (`float64`/`int64`) — no string-to-numeric coercion needed. The one type change this project makes is turning `Class` into a proper binary target named `is_fraud` (Phase 1's naming convention), done in the cleaning step below.

## Invalid values

In [6]:
findings['invalid_values']

{'negative_amount': 0, 'negative_time': 0, 'class_outside_0_1': 0}

No negative transaction amounts, no negative `Time` values, and every `Class` value is 0 or 1. No invalid-value cleaning is required for this dataset — worth stating explicitly rather than silently skipping the check.

## Outliers — investigated, not deleted

In [7]:
findings['amount_outliers_iqr']

{'upper_fence': 184.51,
 'count_above_fence': 31904,
 'pct_above_fence': np.float64(11.2)}

In [8]:
iqr_upper = findings['amount_outliers_iqr']['upper_fence']
is_outlier = df['Amount'] > iqr_upper
fraud_rate_outlier = 100 * df.loc[is_outlier, 'Class'].mean()
fraud_rate_normal = 100 * df.loc[~is_outlier, 'Class'].mean()
print(f"IQR upper fence: {iqr_upper}")
print(f"Fraud rate among Amount outliers:     {fraud_rate_outlier:.4f}%")
print(f"Fraud rate among non-outlier amounts:  {fraud_rate_normal:.4f}%")

IQR upper fence: 184.51
Fraud rate among Amount outliers:     0.2852%
Fraud rate among non-outlier amounts:  0.1586%


**This is exactly the case the phase brief warns about.** Amount outliers (by the standard IQR rule) are *not* obviously more fraudulent here — if anything the fraud rate is comparable or lower among large amounts, likely because large legitimate purchases are common while several known fraud patterns in this dataset cluster at small "testing" amounts. Blindly stripping high-`Amount` rows as "outliers" would have thrown away legitimate high-value transactions without removing fraud signal — exactly the wrong move. These rows are kept.

## Cleaning decision & output

In [9]:
clean_df = preprocessing.clean(df)
print(clean_df.shape)
clean_df.head()

(283726, 32)


,transaction_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,is_fraud
0,TXN0000001,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,TXN0000002,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,TXN0000003,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,TXN0000004,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,TXN0000005,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [10]:
out_dir = Path('..') / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(out_dir / 'transactions_clean.csv', index=False)
print('saved', out_dir / 'transactions_clean.csv')

saved ..\data\processed\transactions_clean.csv


## Summary

| Check | Finding | Action |
|---|---|---|
| Missing values | 0 across all columns | none needed |
| Duplicate rows | 1,081 (19 fraud) | dropped — near-certain data artifacts, and a leakage risk across splits if kept |
| Data types | already numeric | renamed `Class` -> `is_fraud` |
| Invalid values | none (no negative amounts/time, no out-of-range labels) | none needed |
| Amount outliers (IQR) | not associated with higher fraud rate | **kept** — investigated, not removed |

Clean dataset: **283,726 rows** (284,807 - 1,081 duplicates), ready for `03_eda.ipynb`.